# Top2Vec Diversity & Quality using IRBO

This notebook computes **IRBO (Inverted Rank-Biased Overlap)** diversity scores
for every Top2Vec model already trained in `modeling.ipynb`.

**Workflow:**
1. Load saved Top2Vec models for each subject × embedding combination
2. Extract topic words and compute IRBO diversity
3. Save diversity results to `diversity_results_v1.csv`
4. Merge with coherence scores from `coherence_results_v1.csv` to compute **Topic Quality** (harmonic mean)
5. Save combined results to `quality_results_v1.csv`

**Reference:** Bianchi et al. (2021) — *"Pre-training is a Hot Topic"*

In [1]:
import os
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Optional
from itertools import combinations
from tqdm import tqdm
import warnings

from top2vec import Top2Vec

pd.set_option('display.max_colwidth', None)
warnings.filterwarnings("ignore", category=FutureWarning)

## Configuration

In [2]:
VERSION = "v1"
LIST_SUBJECT = ["cs", "math", "physics"]

TRANSFORMERS = [
    "allenai/specter2",
    "sentence-transformers/all-MiniLM-L6-v2",
    "all-distilroberta-v1",
    "intfloat/e5-base-v2",
    "all-mpnet-base-v2",
    "BAAI/bge-base-en-v1.5"
]

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Paths
MODEL_DIR = Path("./transformer")
RESULT_DIR = Path("./transformer")

COHERENCE_CSV = RESULT_DIR / f"coherence_results_{VERSION}.csv"
DIVERSITY_CSV = RESULT_DIR / f"diversity_results_{VERSION}.csv"
QUALITY_CSV = RESULT_DIR / f"quality_results_{VERSION}.csv"

print(f"Coherence results: {COHERENCE_CSV}")
print(f"Diversity results: {DIVERSITY_CSV}")
print(f"Quality results:   {QUALITY_CSV}")
print(f"\nSubjects: {LIST_SUBJECT}")
print(f"Transformers: {len(TRANSFORMERS)}")

Coherence results: transformer/coherence_results_v1.csv
Diversity results: transformer/diversity_results_v1.csv
Quality results:   transformer/quality_results_v1.csv

Subjects: ['cs', 'math', 'physics']
Transformers: 6


## Helper Functions

In [3]:
def get_model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "_").replace("-", "_")


def get_topic_words_top2vec(model: Top2Vec, top_n: int = 10):
    """Extract top-N words for each topic from a Top2Vec model, preserving rank order."""
    num_topics = model.get_num_topics()
    topic_words, _, _ = model.get_topics(num_topics)
    
    topics_words = []
    for i in range(num_topics):
        words = topic_words[i][:top_n].tolist()
        topics_words.append(words)
    
    return topics_words


def rbo(list_1, list_2, p=0.9):
    """
    Rank-Biased Overlap (RBO) between two ranked lists.
    Returns similarity score in [0, 1]. Higher = more similar.
    """
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    
    rbo_score = 0.0
    for d in range(1, k + 1):
        set_1 = set(list_1[:d])
        set_2 = set(list_2[:d])
        agreement = len(set_1 & set_2) / d
        rbo_score += (p ** (d - 1)) * agreement
    
    rbo_score *= (1 - p)
    return rbo_score


def calculate_irbo(topics_words, p=0.9):
    """
    Calculate mean IRBO (Inverted RBO) diversity across all topic pairs.
    Returns mean_irbo in [0, 1]. Higher = more diverse.
    """
    if len(topics_words) < 2:
        return 0.0
    
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    
    return np.mean(irbo_scores)

## Load Coherence Results

In [4]:
coherence_df = pd.read_csv(COHERENCE_CSV)
print(f"Coherence results loaded: {len(coherence_df)} rows")
print(f"Subjects: {coherence_df['subject'].unique().tolist()}")
print(f"\nColumns: {coherence_df.columns.tolist()}")
coherence_df

Coherence results loaded: 18 rows
Subjects: ['cs', 'math', 'physics']

Columns: ['subject', 'model', 'n_topics', 'coherence']


,subject,model,n_topics,coherence
0,cs,allenai/specter2,1,0.297208
1,cs,sentence-transformers/all-MiniLM-L6-v2,808,0.571813
2,cs,all-distilroberta-v1,834,0.480974
3,cs,intfloat/e5-base-v2,1,0.243982
4,cs,all-mpnet-base-v2,1031,0.492803
5,cs,BAAI/bge-base-en-v1.5,92,0.463367
6,math,allenai/specter2,1,0.562219
7,math,sentence-transformers/all-MiniLM-L6-v2,642,0.541479
8,math,all-distilroberta-v1,690,0.466399
9,math,intfloat/e5-base-v2,1,0.309987


## Compute IRBO Diversity

For each subject × embedding combination, load the saved Top2Vec model,
extract topic words, and compute IRBO diversity. **Resume support** is built in —
if `diversity_results_v1.csv` exists, already-computed combinations are skipped.

In [5]:
# Load existing results for resume support
if DIVERSITY_CSV.exists():
    existing_df = pd.read_csv(DIVERSITY_CSV)
    diversity_results = existing_df.to_dict('records')
    completed = set()
    for _, row in existing_df.iterrows():
        key = (row['subject'], row['model'])
        completed.add(key)
    print(f"Resuming: {len(completed)} combinations already completed")
else:
    diversity_results = []
    completed = set()
    print("Starting fresh")

total = len(coherence_df)
remaining = total - len(completed)
print(f"Total combinations: {total}")
print(f"Remaining: {remaining}")

for idx, row in tqdm(coherence_df.iterrows(), total=total, desc="Computing IRBO"):
    subject = row['subject']
    model_name = row['model']
    
    key = (subject, model_name)
    
    # Skip if already computed
    if key in completed:
        continue
    
    # Build model path
    safe_name = get_model_safe_name(model_name)
    model_path = str(MODEL_DIR / subject / f"{safe_name}_{VERSION}")
    
    if not os.path.exists(model_path):
        print(f"  ⚠ Model not found: {model_path}")
        continue
    
    try:
        # Load saved Top2Vec model
        model = Top2Vec.load(model_path)
        n_topics = model.get_num_topics()
        
        # Extract topic words and compute IRBO
        topics_words = get_topic_words_top2vec(model, top_n=TOP_N_WORDS)
        irbo_mean = calculate_irbo(topics_words, p=RBO_P)
        
        result = {
            "subject": subject,
            "model": model_name,
            "n_topics": n_topics,
            "irbo_mean": irbo_mean,
        }
        
        diversity_results.append(result)
        completed.add(key)
        
        # Incremental save
        pd.DataFrame(diversity_results).to_csv(DIVERSITY_CSV, index=False)
        
        del model
        gc.collect()
        
    except Exception as e:
        print(f"  ✗ Error [{subject} / {model_name}]: {e}")
        continue

print(f"\n{'='*60}")
print(f"IRBO diversity computation complete!")
print(f"Results saved to: {DIVERSITY_CSV}")
print(f"Total results: {len(diversity_results)}")

Starting fresh
Total combinations: 18
Remaining: 18


Computing IRBO: 100%|██████████| 18/18 [00:38<00:00,  2.16s/it]


IRBO diversity computation complete!
Results saved to: transformer/diversity_results_v1.csv
Total results: 18


## Diversity Results

In [6]:
diversity_df = pd.read_csv(DIVERSITY_CSV)
print(f"Diversity results: {len(diversity_df)} rows")
diversity_df

Diversity results: 18 rows


,subject,model,n_topics,irbo_mean
0,cs,allenai/specter2,1,0.000000
1,cs,sentence-transformers/all-MiniLM-L6-v2,808,0.993782
2,cs,all-distilroberta-v1,834,0.982417
3,cs,intfloat/e5-base-v2,1,0.000000
4,cs,all-mpnet-base-v2,1031,0.984995
5,cs,BAAI/bge-base-en-v1.5,92,0.987560
6,math,allenai/specter2,1,0.000000
7,math,sentence-transformers/all-MiniLM-L6-v2,642,0.992714
8,math,all-distilroberta-v1,690,0.989079
9,math,intfloat/e5-base-v2,1,0.000000


## Merge with Coherence & Compute Topic Quality

$$\text{Topic Quality} = 2 \times \frac{\text{Coherence} \times \text{IRBO}}{\text{Coherence} + \text{IRBO}}$$

In [7]:
# Load results
coherence_df = pd.read_csv(COHERENCE_CSV)
diversity_df = pd.read_csv(DIVERSITY_CSV)

print(f"Coherence results: {len(coherence_df)} rows")
print(f"Diversity results: {len(diversity_df)} rows")

# Merge on subject + model
merge_cols = ['subject', 'model']
combined_df = coherence_df.merge(
    diversity_df[merge_cols + ['irbo_mean']],
    on=merge_cols,
    how='inner'
)

# Topic Quality = harmonic mean of coherence and IRBO
combined_df['topic_quality'] = (
    2 * combined_df['coherence'] * combined_df['irbo_mean'] /
    (combined_df['coherence'] + combined_df['irbo_mean'])
)

# Save
combined_df.to_csv(QUALITY_CSV, index=False)

print(f"\nCombined results: {len(combined_df)} rows")
print(f"Saved to: {QUALITY_CSV}")

Coherence results: 18 rows
Diversity results: 18 rows

Combined results: 18 rows
Saved to: transformer/quality_results_v1.csv


## Quality Results Table

In [8]:
print("\n" + "=" * 110)
print("COMBINED TOPIC QUALITY RESULTS (Coherence × IRBO)")
print("=" * 110)
print(
    f"{'Subject':<10} "
    f"{'Model':<48} "
    f"{'#Topics':>8} {'Coherence':>10} {'IRBO':>10} {'Quality':>10}"
)
print("-" * 110)

for subject in LIST_SUBJECT:
    subject_data_df = combined_df[combined_df['subject'] == subject].sort_values(
        'topic_quality', ascending=False
    )
    for _, row in subject_data_df.iterrows():
        print(
            f"{row['subject']:<10} "
            f"{row['model']:<48} "
            f"{int(row['n_topics']):>8} "
            f"{row['coherence']:>10.4f} "
            f"{row['irbo_mean']:>10.6f} "
            f"{row['topic_quality']:>10.4f}"
        )
    print("-" * 110)

print("=" * 110)


COMBINED TOPIC QUALITY RESULTS (Coherence × IRBO)
Subject    Model                                             #Topics  Coherence       IRBO    Quality
--------------------------------------------------------------------------------------------------------------
cs         sentence-transformers/all-MiniLM-L6-v2                808     0.5718   0.993782     0.7259
cs         all-mpnet-base-v2                                    1031     0.4928   0.984995     0.6569
cs         all-distilroberta-v1                                  834     0.4810   0.982417     0.6458
cs         BAAI/bge-base-en-v1.5                                  92     0.4634   0.987560     0.6308
cs         allenai/specter2                                        1     0.2972   0.000000     0.0000
cs         intfloat/e5-base-v2                                     1     0.2440   0.000000     0.0000
--------------------------------------------------------------------------------------------------------------
math       se

## Best Model per Subject

In [9]:
print("\n" + "=" * 100)
print("BEST MODEL PER SUBJECT (by Topic Quality)")
print("=" * 100)

best_models = {}

for subject in LIST_SUBJECT:
    subject_df = combined_df[combined_df['subject'] == subject]
    if len(subject_df) == 0 or subject_df['topic_quality'].isna().all():
        print(f"\n  {subject.upper()}: No results")
        continue
    
    best = subject_df.loc[subject_df['topic_quality'].idxmax()]
    best_models[subject] = best
    
    print(f"\n  {subject.upper()}:")
    print(f"    Model:      {best['model']}")
    print(f"    Coherence:  {best['coherence']:.4f}")
    print(f"    IRBO Mean:  {best['irbo_mean']:.6f}")
    print(f"    Quality:    {best['topic_quality']:.4f}")
    print(f"    # Topics:   {int(best['n_topics'])}")

print("\n" + "=" * 100)

# Overall best
if len(combined_df) > 0:
    best_overall = combined_df.loc[combined_df['topic_quality'].idxmax()]
    print(f"\n★ OVERALL BEST: {best_overall['subject']} "
          f"({best_overall['model']}) "
          f"with Quality = {best_overall['topic_quality']:.4f}")
    print(f"  (Coherence={best_overall['coherence']:.4f}, IRBO={best_overall['irbo_mean']:.6f})")


BEST MODEL PER SUBJECT (by Topic Quality)

  CS:
    Model:      sentence-transformers/all-MiniLM-L6-v2
    Coherence:  0.5718
    IRBO Mean:  0.993782
    Quality:    0.7259
    # Topics:   808

  MATH:
    Model:      sentence-transformers/all-MiniLM-L6-v2
    Coherence:  0.5415
    IRBO Mean:  0.992714
    Quality:    0.7007
    # Topics:   642

  PHYSICS:
    Model:      sentence-transformers/all-MiniLM-L6-v2
    Coherence:  0.6354
    IRBO Mean:  0.993754
    Quality:    0.7752
    # Topics:   687


★ OVERALL BEST: physics (sentence-transformers/all-MiniLM-L6-v2) with Quality = 0.7752
  (Coherence=0.6354, IRBO=0.993754)


## Quality Score Pivot Table

In [10]:
if len(combined_df) > 0 and combined_df['topic_quality'].notna().any():
    valid = combined_df.dropna(subset=['topic_quality'])
    pivot = valid.pivot(index='model', columns='subject', values='topic_quality')
    pivot['mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean', ascending=False)
    print("\nTopic Quality by Model × Subject:\n")
    print(pivot.round(4))


Topic Quality by Model × Subject:

subject                                     cs    math  physics    mean
model                                                                  
sentence-transformers/all-MiniLM-L6-v2  0.7259  0.7007   0.7752  0.7340
all-mpnet-base-v2                       0.6569  0.6470   0.7122  0.6721
all-distilroberta-v1                    0.6458  0.6339   0.6230  0.6342
BAAI/bge-base-en-v1.5                   0.6308  0.6050   0.6186  0.6181
allenai/specter2                        0.0000  0.0000   0.0000  0.0000
intfloat/e5-base-v2                     0.0000  0.0000   0.0000  0.0000
